# 03 — Reproduce CellNEST graph construction

This notebook reproduces the **graph-construction stage** of
[CellNEST](https://github.com/schwartzlab-methods/CellNEST) (Fatema *et al.*, *Nature
Methods* 2025) using our clean-room package [`src/cellnest_graph`](../src/cellnest_graph).

It **stops at graph construction** — no GAT training, no Deep Graph Infomax, no higher-order
lifting. The output is a backend-neutral graph ready to be lifted to a TopoNetX cell complex
in the next milestone.

See [`docs/cellnest_graph_reference.md`](../docs/cellnest_graph_reference.md) for the exact
trace of the original CellNEST code.

## 1. The CellNEST graph definition (plain English)

**Node** = one spatial cell/spot. Node feature = its (quantile-normalised) gene-expression
vector, or a reduced representation (e.g. one-hot cell type). We keep cell id, coordinates,
sample id and cell type when available.

**Directed typed edge `i → j`** for a ligand–receptor pair `(l, r)` exists when:

1. cell `i` expresses ligand `l` (active);
2. cell `j` expresses receptor `r` (active);
3. `distance(i, j) ≤ d_max` (spatial neighbourhood);
4. `(l, r)` is in the supplied LR database.

Each `(l, r)` pair is a **distinct relation type**. Edge attributes: source/target,
ligand, receptor, relation id, Euclidean distance, ligand expression in the sender, receptor
expression in the receiver, co-expression score `= expr_l(i)·expr_r(j)`, a distance weight,
and an optional distance-modulated score.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cellnest_graph import build_cellnest_graph
from cellnest_graph.synthetic import toy_dataset
print("imports OK")

## 2. Run the deterministic synthetic example

Six cells; two ligands, two receptors; three LR relations. Cells 4 and 5 are placed far away
so they are spatially isolated at `d_max = 1.5`. The expected directed edges are known in
advance (encoded in `toy_dataset().expected_edges`).

In [ ]:
ds = toy_dataset()
graph = build_cellnest_graph(
    ds.adata, ds.lr_pairs,
    d_max=ds.d_max,
    gene_activity_percentile=None,   # toy example uses absolute thresholds for determinism
    block_autocrine=True,
)
print("nodes:", graph.n_nodes, " edges:", graph.n_edges, " relations:", graph.n_relations)

# check we reproduced the hand-derived edge set exactly
got = {(int(r.source), int(r.target), r.ligand, r.receptor, int(r.relation_id))
       for r in graph.edge_table.itertuples(index=False)}
exp = {(s,t,l,rc,rid) for (s,t,l,rc,rid,_d,_c) in ds.expected_edges}
print("edge set matches expected:", got == exp)

## 3. Visualise cells and directed typed edges

In [ ]:
coords = graph.coordinates
rel_colors = plt.cm.tab10(np.linspace(0, 1, max(graph.n_relations, 1)))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=400, c="#dddddd", edgecolors="k", zorder=3)
for i, (x, y) in enumerate(coords):
    ax.annotate(f"c{i}", (x, y), ha="center", va="center", zorder=4, fontweight="bold")

for r in graph.edge_table.itertuples(index=False):
    s, t = int(r.source), int(r.target)
    if s == t:
        continue
    x0, y0 = coords[s]; x1, y1 = coords[t]
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=rel_colors[int(r.relation_id)],
                                lw=1.8, shrinkA=16, shrinkB=16,
                                connectionstyle="arc3,rad=0.12"), zorder=2)

handles = [plt.Line2D([0], [0], color=rel_colors[i], lw=2,
                      label=f"rel {i}: {row.ligand}->{row.receptor}")
           for i, row in graph.relation_table.iterrows()]
ax.legend(handles=handles, loc="upper left", fontsize=9)
ax.set_title("CellNEST-style directed typed LR graph (toy)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## 4. Node, edge and relation tables

In [ ]:
try:
    from IPython.display import display
except Exception:            # running outside Jupyter
    display = lambda df: print(df.to_string(index=False))
print("Relation mapping:"); display(graph.relation_table)
print("\nNodes:");          display(graph.node_table)
print("\nEdges:");          display(graph.edge_table)

## 5. Graph statistics

In [ ]:
for k, v in graph.stats().items():
    print(f"{k:28s}: {v}")

print("\nedge feature columns:", graph.edge_feature_names)
print("edge_index shape:", graph.edge_index.shape)
print("edge_features shape:", graph.edge_features.shape)
print("node_features shape:", graph.node_features.shape)

## 6. (Optional) small real AnnData subset

This block runs **only if** a local `.h5ad` already exists — it never downloads the full
dataset. It uses `--max-cells`-style capping and a single sample so it stays a smoke test.
Edit `H5AD` / `SAMPLE_KEY` to point at your file.

In [ ]:
from pathlib import Path
from cellnest_graph import load_lr_pairs_csv

H5AD = os.environ.get("ST_DATA")  # or set a path string here
candidates = sorted(Path("../data").glob("*.h5ad"))
if H5AD is None and candidates:
    H5AD = str(candidates[0])

if H5AD and os.path.exists(H5AD):
    import scanpy as sc
    adata = sc.read_h5ad(H5AD)
    print("loaded", H5AD, adata.shape)
    lr = load_lr_pairs_csv("../data/ligand_receptor_pairs.csv")
    # cap to a small subset for the smoke test
    sub = adata[:2000].copy()
    g = build_cellnest_graph(sub, lr, spatial_key="spatial", d_max=30.0,
                             gene_activity_percentile=98.0, max_cells=2000)
    for k, v in g.stats().items():
        print(f"{k:28s}: {v}")
else:
    print("No local .h5ad found — skipping the real-data demo (this is expected).")
    print("See data/datasets.md to download a dataset into data/.")

## 7. Next step (do **not** run here)

The neutral `CellNestGraph` is designed to be lifted to a higher-order structure:

```
graph  ->  TopoNetX cell / simplicial complex  (0-cells = nodes, 1-cells = edges, 2-cells = filled motifs)
```

That lifting is the **next milestone** and is intentionally not started in this notebook.
`graph.to_networkx()` and `src/topo_utils.py::graph_to_simplices` are the natural entry
points for it.